In [ ]:
%pip install -q langchain langchain-aws langgraph pydantic boto3 python-dotenv chromadb langchain-chroma

# Week 4 Friday — Structured Output, Error Handling & RAG Tools

Today we close out the LangChain core block (Wednesday through Friday) before pivoting to **LangSmith observability** on Monday. The theme is *production readiness*: getting structured data out of models, handling the errors that inevitably occur, and wiring vector stores into agents so they can retrieve knowledge on demand.

## Learning Objectives

1. Use **Pydantic** models with `with_structured_output()` to get typed, validated responses from an LLM.
2. Implement **error-handling patterns** — retry with backoff, fallback mechanisms, and graceful degradation — so agents don't crash in production.
3. Build **RAG tool agents** that wrap a ChromaDB vector store in a `@tool` function, letting the agent decide *when* to retrieve.

## Roadmap

| Stage | Topic | Key Idea |
|-------|-------|----------|
| **1** | Structured Output with Pydantic | `with_structured_output()` converts free-form text into typed Python objects |
| **2** | Error Handling Patterns | Catch errors inside tools, retry transient failures, fall back gracefully |
| **3** | RAG Tool Agents | Wrap vector-store search in `@tool` — the agent decides when to retrieve |

---

# Stage 1 — Structured Output with Pydantic

Without structured output, LLM responses are just strings. Extracting specific fields means writing fragile parsing code that breaks whenever the model rephrases its answer. **Pydantic + `with_structured_output()`** solves this: you define a schema, and LangChain guarantees the model returns an instance of that schema — a real Python object with typed attributes.

### What we'll cover

1. The problem with unstructured output
2. Defining Pydantic schemas with `Field` descriptions
3. Using `with_structured_output()` for typed responses
4. Nested schemas for complex extraction
5. Structured output with agents (`response_format`)
6. Handling validation errors

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")

# --- PART 1: The problem with unstructured output ---

review = "Great laptop, fast shipping, but battery life is poor. Screen is amazing!"
response = model.invoke(f"Analyze this product review: '{review}'")

print("Raw unstructured response:")
print(response.content[:300])
print("\nProblem: how do we reliably extract sentiment, pros, cons, and a rating from free text?")

### Defining a Pydantic Schema

A **Pydantic model** declares the exact shape of data we want back. Each field has a Python type and a `description` that guides the LLM.

```python
class ReviewAnalysis(BaseModel):
    sentiment: Sentiment        # enum constrains allowed values
    rating_estimate: int        # Field(ge=1, le=5) adds range validation
    pros: List[str]             # list of positive points
    cons: List[str]             # list of negative points
    summary: str                # one-sentence summary
```

When we call `model.with_structured_output(ReviewAnalysis)`, LangChain instructs the LLM to return JSON matching this schema and then deserialises it into a real `ReviewAnalysis` instance — with full attribute access, type safety, and automatic validation.

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from enum import Enum


class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    MIXED = "mixed"
    NEUTRAL = "neutral"


class ReviewAnalysis(BaseModel):
    sentiment: Sentiment = Field(description="Overall sentiment of the review")
    rating_estimate: int = Field(ge=1, le=5, description="Estimated star rating 1-5")
    pros: List[str] = Field(description="Positive aspects mentioned")
    cons: List[str] = Field(description="Negative aspects mentioned")
    summary: str = Field(description="Brief one-sentence summary")


structured_model = model.with_structured_output(ReviewAnalysis)

review = "Great laptop, fast shipping, but battery life is poor. Screen is amazing!"
result = structured_model.invoke(f"Analyze this product review: '{review}'")

print(f"Type      : {type(result).__name__}")
print(f"Sentiment : {result.sentiment.value}")
print(f"Rating    : {result.rating_estimate}/5")
print(f"Pros      : {result.pros}")
print(f"Cons      : {result.cons}")
print(f"Summary   : {result.summary}")

### Complex Nested Schemas

Pydantic models can reference other models. This is perfect for extracting structured data from messy, real-world text like emails — the outer model describes the whole meeting request while an inner model describes each attendee.

In [ ]:
class ContactInfo(BaseModel):
    name: Optional[str] = Field(default=None, description="Person's name")
    email: Optional[str] = Field(default=None, description="Email address")
    company: Optional[str] = Field(default=None, description="Company name")


class MeetingRequest(BaseModel):
    attendees: List[ContactInfo] = Field(description="People to attend the meeting")
    proposed_date: Optional[str] = Field(default=None, description="Suggested date")
    proposed_time: Optional[str] = Field(default=None, description="Suggested time")
    topic: str = Field(description="Meeting topic or purpose")
    priority: str = Field(description="Priority level: low, medium, or high")


meeting_model = model.with_structured_output(MeetingRequest)

email_text = """
Hey team,

Can we set up a call to discuss the Q4 roadmap? I'm thinking next Tuesday at 2pm.
Please include John Smith (john@company.com) and Sarah Johnson from marketing.
This is pretty urgent - we need to finalize before the board meeting.

Thanks!
"""

meeting = meeting_model.invoke(f"Extract meeting request details from this email:\n\n{email_text}")

print(f"Topic    : {meeting.topic}")
print(f"Priority : {meeting.priority}")
print(f"Date     : {meeting.proposed_date}")
print(f"Time     : {meeting.proposed_time}")
print("Attendees:")
for a in meeting.attendees:
    print(f"  - {a.name} | {a.email} | {a.company}")

### Structured Output with Agents

You can combine tools **and** structured output by passing `response_format` to `create_agent`. The agent uses its tools normally, then its final answer is forced into the Pydantic schema you specify.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


class Product(BaseModel):
    product_name: str = Field(description="Name of product")
    in_stock: int = Field(description="Units currently in stock")
    on_order: int = Field(description="Units on order")


@tool
def lookup_inventory(product_name: str) -> str:
    """Look up product inventory levels."""
    inventory = {
        "laptop": "42 units in stock, 10 on order",
        "monitor": "0 units in stock, 50 on order",
        "keyboard": "128 units in stock, 0 on order",
    }
    return inventory.get(product_name.lower(), f"Product '{product_name}' not found")


inventory_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[lookup_inventory],
    system_prompt="You are an inventory assistant. Use lookup_inventory to find product information.",
    checkpointer=InMemorySaver(),
    response_format=Product,
    name="structured_inventory_agent",
)

result = inventory_agent.invoke(
    {"messages": [{"role": "user", "content": "Check inventory for laptop"}]},
    {"configurable": {"thread_id": "inv_001"}},
)

print("Agent response:")
print(result["messages"][-1].content)

### Handling Validation Errors

Pydantic validates data automatically. If the LLM returns a value outside the allowed range — or a field is missing — you get a `ValidationError`. Always wrap structured-output calls in `try/except` when your schemas have tight constraints.

In [ ]:
from pydantic import ValidationError


class StrictRating(BaseModel):
    score: int = Field(ge=1, le=5, description="Rating score 1-5")
    reason: str = Field(min_length=10, description="Reason for the rating (at least 10 chars)")


print("--- Pydantic validation in action ---")

try:
    valid = StrictRating(score=4, reason="Great product with fast delivery!")
    print(f"Accepted : score={valid.score}, reason='{valid.reason}'")
except ValidationError as e:
    print(f"Rejected : {e}")

try:
    invalid = StrictRating(score=10, reason="Bad")
    print(f"Accepted : score={invalid.score}")
except ValidationError as e:
    print(f"Rejected (expected): score=10 is out of range and reason is too short")

print("\n--- Structured output through the model ---")

rating_model = model.with_structured_output(StrictRating)
result = rating_model.invoke("Rate the following product: 'Decent headphones, nothing special.'")

print(f"Score  : {result.score}")
print(f"Reason : {result.reason}")

---

# Stage 2 — Error Handling Patterns

Production agents encounter errors constantly: API rate limits, tool failures, malformed inputs, unexpected model output. A single unhandled exception can crash the entire system. This stage covers three defensive patterns that keep your agents running:

1. **Tool-level error handling** — catch exceptions *inside* the tool and return an error string instead of crashing the agent.
2. **Retry with exponential backoff** — re-attempt transient failures with increasing delays (1 s → 2 s → 4 s).
3. **Fallback mechanisms** — when the primary service fails, switch to a backup.

In [ ]:
import time
import random


@tool
def bad_api_call(endpoint: str) -> str:
    """Call an external API — no error handling!"""
    import requests
    return requests.get(endpoint).text


print("--- Unhandled error (bad tool) ---")
try:
    bad_api_call.invoke({"endpoint": "https://this-domain-does-not-exist-12345.com"})
except Exception as e:
    print(f"Crash! {type(e).__name__}")
    print(f"If this tool were inside an agent, the whole agent would die.\n")


@tool
def good_api_call(endpoint: str) -> str:
    """Call an external API safely. Returns an error message on failure."""
    import requests
    try:
        resp = requests.get(endpoint, timeout=5)
        resp.raise_for_status()
        return resp.text[:500]
    except requests.exceptions.Timeout:
        return "ERROR: request timed out."
    except requests.exceptions.ConnectionError:
        return "ERROR: could not connect to the service."
    except requests.exceptions.HTTPError as e:
        return f"ERROR: HTTP {e.response.status_code}"
    except Exception as e:
        return f"ERROR: {type(e).__name__}"


print("--- Handled error (good tool) ---")
result = good_api_call.invoke({"endpoint": "https://this-domain-does-not-exist-12345.com"})
print(f"Tool returned: {result}")
print("The agent stays alive and can explain the error to the user.")

### Retry with Exponential Backoff

Transient failures — rate limits, brief network blips, database hiccups — often resolve themselves within seconds. A retry loop with exponentially increasing delays handles these gracefully without hammering the service.

The formula is simple: `delay = base_delay * 2^attempt`. With a 1-second base that gives waits of 1 s, 2 s, 4 s before giving up.

In [ ]:
@tool
def flaky_database_query(query: str) -> str:
    """Query a database that fails roughly 30 percent of the time."""
    if random.random() < 0.3:
        raise ConnectionError("Database connection lost")
    time.sleep(0.1)
    return f"Result for '{query}': [sample data returned successfully]"


def retry_with_backoff(func, args: dict, max_retries: int = 3, base_delay: float = 1.0) -> str:
    last_error = None
    for attempt in range(max_retries):
        try:
            return func.invoke(args)
        except Exception as e:
            last_error = str(e)
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"  Attempt {attempt + 1} failed — retrying in {delay}s ...")
                time.sleep(delay)
    return f"FAILED after {max_retries} attempts. Last error: {last_error}"


print("--- Retry with backoff ---")
result = retry_with_backoff(flaky_database_query, {"query": "SELECT * FROM orders"})
print(f"Final result: {result}")

### Fallback Mechanisms

When the *primary* service is down, switch to a *backup*. The backup might be slower or return less-detailed results, but it keeps the system functional. This is the same pattern used in production systems everywhere — primary database fails, read from a replica; primary API is down, call the secondary provider.

In [ ]:
@tool
def primary_search(query: str) -> str:
    """Fast primary search engine (sometimes unavailable)."""
    if random.random() < 0.5:
        return "ERROR: primary search unavailable."
    return f"Primary results for '{query}': fast, accurate results."


@tool
def backup_search(query: str) -> str:
    """Slower but reliable backup search engine."""
    time.sleep(0.3)
    return f"Backup results for '{query}': reliable results from backup."


def search_with_fallback(query: str) -> str:
    result = primary_search.invoke({"query": query})
    if not result.startswith("ERROR:"):
        return f"[Primary] {result}"
    print("  Primary failed -> trying backup ...")
    result = backup_search.invoke({"query": query})
    return f"[Backup] {result}"


print("--- Fallback pattern (run a few times to see both paths) ---")
for i in range(3):
    print(f"\nAttempt {i + 1}:")
    print(f"  {search_with_fallback('machine learning tutorials')}")

### Putting It Together: A Robust Agent

Now let's wire error-handling tools into an actual agent. Because every tool returns a *string* on failure (instead of raising an exception), the agent stays alive and can explain the problem to the user naturally.

In [ ]:
@tool
def reliable_product_lookup(product_id: str) -> str:
    """Look up product info. Returns a clear error message for invalid IDs."""
    if not product_id or len(product_id) < 2:
        return "ERROR: invalid product ID. Use format 'P001'."
    products = {
        "P001": {"name": "Laptop Pro", "price": 999.99, "stock": 42},
        "P002": {"name": "Wireless Mouse", "price": 29.99, "stock": 156},
        "P003": {"name": "USB-C Hub", "price": 49.99, "stock": 0},
    }
    product = products.get(product_id.upper())
    if not product:
        return f"ERROR: product '{product_id}' not found. Available: {list(products.keys())}"
    status = "In Stock" if product["stock"] > 0 else "Out of Stock"
    return f"{product['name']} - ${product['price']:.2f} - {status} ({product['stock']} units)"


@tool
def safe_calculation(expression: str) -> str:
    """Evaluate a basic math expression safely."""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "ERROR: only numbers and basic operators (+, -, *, /) are allowed."
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except ZeroDivisionError:
        return "ERROR: division by zero."
    except Exception as e:
        return f"ERROR: {type(e).__name__}"


robust_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[reliable_product_lookup, safe_calculation],
    system_prompt=(
        "You are a shopping assistant. Use the available tools to help customers. "
        "If a tool returns an ERROR, explain it clearly and suggest alternatives."
    ),
    checkpointer=InMemorySaver(),
    name="robust_shopping_agent",
)

config = {"configurable": {"thread_id": "robust_demo"}}

queries = [
    "What's the price of P001?",
    "Look up product INVALID",
    "Calculate 100 * 0.15",
    "Calculate 100 / 0",
]

for q in queries:
    print(f"\nUser : {q}")
    result = robust_agent.invoke(
        {"messages": [{"role": "user", "content": q}]}, config
    )
    print(f"Agent: {result['messages'][-1].content[:200]}")

---

# Stage 3 — RAG Tool Agents

RAG (Retrieval-Augmented Generation) connects agents to a **knowledge base**. Instead of relying only on training data, the agent can search a vector store for relevant documents before answering.

In **agentic RAG**, the agent *decides* whether retrieval is needed — a greeting doesn't require a database lookup, but a policy question does. This is more efficient than traditional 2-step RAG, which retrieves on every single query.

### What we'll build

1. A local **ChromaDB** vector store loaded with sample company documents
2. A `@tool`-decorated retrieval function the agent can call
3. An agent that uses the tool to answer customer-support questions

In [ ]:
from langchain_aws import BedrockEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")

docs = [
    Document(
        page_content=(
            "REFUND POLICY: Customers can request a full refund within 30 days "
            "of purchase for any unused product. Used products may be eligible for "
            "partial refund at management discretion. Digital products are "
            "non-refundable after download. Refunds are processed within 5-7 "
            "business days to the original payment method."
        ),
        metadata={"category": "policies"},
    ),
    Document(
        page_content=(
            "SHIPPING INFORMATION: Standard shipping takes 5-7 business days. "
            "Express shipping (2-3 days) available for $12.99. Free shipping on "
            "orders over $50. International shipping available to select countries "
            "with additional customs fees."
        ),
        metadata={"category": "logistics"},
    ),
    Document(
        page_content=(
            "WARRANTY COVERAGE: All electronics carry a 1-year manufacturer "
            "warranty covering defects in materials and workmanship. Extended "
            "warranty available (2 or 3 year options). Does not cover accidental "
            "damage, water damage, or misuse. Warranty claims require proof of purchase."
        ),
        metadata={"category": "policies"},
    ),
    Document(
        page_content=(
            "ACCOUNT MANAGEMENT: Users can update profile information in "
            "Settings > Profile. Password reset available via email verification. "
            "Two-factor authentication recommended. Account deletion requests "
            "processed within 48 hours."
        ),
        metadata={"category": "support"},
    ),
    Document(
        page_content=(
            "PAYMENT OPTIONS: We accept Visa, Mastercard, American Express, "
            "and PayPal. Apple Pay and Google Pay available on mobile. Monthly "
            "payment plans available for orders over $200 through Affirm. "
            "Gift cards accepted online and in-store."
        ),
        metadata={"category": "payments"},
    ),
]

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)

print(f"Indexed {len(docs)} documents into ChromaDB (in-memory).")

test_results = vectorstore.similarity_search("refund policy", k=2)
for i, d in enumerate(test_results):
    print(f"\n  Doc {i + 1}: {d.page_content[:80]}...")

### Creating the RAG Tool

The `@tool` decorator turns a regular function into something an agent can call. The **docstring is critical** — it tells the agent *when* this tool is appropriate. A vague docstring leads to the agent searching when it shouldn't (or not searching when it should).

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the company knowledge base for relevant information.

    Use this tool when the user asks about:
    - Refund or return policies
    - Shipping information and delivery times
    - Warranty coverage and claims
    - Account settings and management
    - Payment methods and options

    Returns relevant documentation to answer customer questions.
    """
    results = vectorstore.similarity_search(query, k=2)
    if not results:
        return "No relevant information found in the knowledge base."
    formatted = []
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get("category", "unknown").upper()
        formatted.append(f"[Source: {source}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)


rag_agent = create_agent(
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a customer-support agent for an e-commerce company.\n\n"
        "When customers ask about policies, shipping, warranties, accounts, or payments:\n"
        "1. Use search_knowledge_base to find relevant information.\n"
        "2. Synthesize the results into a friendly, conversational answer.\n"
        "3. If the knowledge base doesn't have the answer, say so honestly.\n\n"
        "For greetings or general chat, respond directly without searching."
    ),
    checkpointer=InMemorySaver(),
    name="rag_support_agent",
)

print("RAG agent created.")

### Testing the RAG Agent

Let's send a few customer questions. The agent will search the knowledge base, pull back the relevant documents, and synthesize a conversational answer.

In [ ]:
config = {"configurable": {"thread_id": "support_session_001"}}

questions = [
    "Can I get a refund if I already opened the product?",
    "How long does shipping take?",
    "What does the warranty cover?",
]

for q in questions:
    print(f"Customer: {q}")
    result = rag_agent.invoke(
        {"messages": [{"role": "user", "content": q}]}, config
    )
    print(f"Agent   : {result['messages'][-1].content[:300]}\n")

### 2-Step RAG vs. Agentic RAG

| | 2-Step RAG | Agentic RAG |
|---|---|---|
| Retrieval | **Always** happens | Agent **decides** |
| Query count | Single fixed query | Can reformulate or follow up |
| Flow | Retrieve → Generate | Think → Retrieve? → Generate → Repeat? |
| Latency | Lower (one retrieval) | Potentially higher |
| Best for | Simple FAQ / doc QA | Multi-turn, mixed conversations |

The cell below demonstrates the *agentic* side — the agent skips the search tool entirely when retrieval isn't needed.

In [ ]:
print("--- Agentic RAG: agent decides NOT to search ---\n")

greeting = "Hello! How are you today?"
print(f"Customer: {greeting}")

result = rag_agent.invoke(
    {"messages": [{"role": "user", "content": greeting}]},
    {"configurable": {"thread_id": "agentic_demo"}},
)

print(f"Agent   : {result['messages'][-1].content}")
print("\n(The agent responded without calling search_knowledge_base — no retrieval needed.)")

---

# Key Takeaways

### This session (Friday)

| Concept | One-liner |
|---------|-----------|
| `with_structured_output()` | Attach a Pydantic model to get typed, validated responses |
| `Field(description=...)` | Descriptions guide the LLM on what each field means |
| Nested schemas | Pydantic models can reference other models for complex extraction |
| Tool error handling | Return error *strings* from tools — never let exceptions crash the agent |
| Retry with backoff | `delay = base * 2^attempt` handles transient failures |
| Fallback pattern | Primary → backup → default keeps the system functional |
| RAG tool | Wrap `vectorstore.similarity_search` in `@tool` |
| Agentic RAG | The agent decides *when* to retrieve — not every query needs it |

### The full LangChain block (Wednesday → Friday)

- **Wednesday** — Chat models via `init_chat_model`, tool creation with `@tool`, building agents with `create_agent` + LangGraph
- **Thursday** — Memory and state: `InMemorySaver`, conversation threads, system prompts
- **Friday** — Structured output, error handling, RAG tool agents

### Up next: Week 5 Monday — LangSmith Observability

Next week we connect our agents to **LangSmith** to get full visibility into every LLM call, tool invocation, and token cost. Tracing is the foundation of debugging and evaluating production agents — you can't improve what you can't see.